In [ ]:
## Setup — packages & environment

import sys, subprocess
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'networkx', 'openpyxl', 'reportlab']
for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except:
        install(pkg)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
from scipy import stats
from scipy.linalg import inv
import os, datetime as dt

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Survey Data Creation

# Synthetic psychological survey data
np.random.seed(RSEED)
n_respondents = 200

variables = ['Motivation', 'SelfEfficacy', 'CourseEngagement', 'Satisfaction', 'GPA']

# Create correlated data
data = np.random.multivariate_normal(
    mean=[50, 60, 55, 70, 75],
    cov=[
        [100, 80, 60, 40, 50],
        [80, 120, 70, 50, 60],
        [60, 70, 110, 65, 70],
        [40, 50, 65, 130, 80],
        [50, 60, 70, 80, 100]
    ],
    size=n_respondents
)

df = pd.DataFrame(data, columns=variables)
print('Survey Data:')
print(df.describe())

In [ ]:
## 2. Correlation and Partial Correlation

# Correlation matrix
corr_matrix = df.corr()

# Partial correlation via precision matrix
cov_matrix = df.cov()
precision_matrix = inv(cov_matrix)

# Normalize precision to partial correlations
partial_corr = np.zeros_like(precision_matrix)
for i in range(len(precision_matrix)):
    for j in range(len(precision_matrix)):
        if i != j:
            partial_corr[i, j] = -precision_matrix[i, j] / np.sqrt(precision_matrix[i, i] * precision_matrix[j, j])
        else:
            partial_corr[i, j] = 1

partial_corr_df = pd.DataFrame(partial_corr, index=variables, columns=variables)

print('Correlation Matrix:')
print(corr_matrix.round(3))
print('\nPartial Correlation Matrix:')
print(partial_corr_df.round(3))

In [ ]:
## 3. Network Construction from Partial Correlations

# Create network with threshold
threshold = 0.1
G = nx.Graph()

for i, var1 in enumerate(variables):
    for j, var2 in enumerate(variables):
        if i < j and abs(partial_corr[i, j]) > threshold:
            G.add_edge(var1, var2, weight=partial_corr[i, j])

# Add isolated nodes
for var in variables:
    if var not in G:
        G.add_node(var)

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print('\nEdges:')
for u, v, data in G.edges(data=True):
    print(f'{u} - {v}: {data["weight"]:.3f}')

os.makedirs('figures', exist_ok=True)

# 1. Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/01_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_correlation_heatmap.png')

# 2. Partial correlation heatmap
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(partial_corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Partial Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/02_partial_correlation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_partial_correlation.png')

# 3. Psychological network
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G, k=2, iterations=50, seed=RSEED)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='lightblue', ax=ax, alpha=0.8)

# Draw edges with colors based on positive/negative
positive_edges = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] > 0]
negative_edges = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] < 0]

nx.draw_networkx_edges(G, pos, edgelist=positive_edges, edge_color='green', width=2, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=negative_edges, edge_color='red', width=2, style='dashed', ax=ax)

nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)

ax.set_title('Psychological Network (Partial Correlation)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/03_psychological_network.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_psychological_network.png')

df.to_csv('psychological_survey_data.csv', index=False)
print('\nSaved survey data')

In [ ]:
## 5. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch19_PsychologicalNetworks_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 19: Psychological Networks</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Psychological networks represent latent relationships between psychological constructs. '
    'Partial correlations reveal direct associations after controlling for other variables, '
    'distinguishing direct effects from indirect effects mediated through other variables.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Variables</b>', styles['Heading2']))
vars_text = f'Measured constructs: {', '.join(variables)} (N={n_respondents} respondents).'
story.append(Paragraph(vars_text, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Methods</b>', styles['Heading2']))
methods = (
    'Pearson correlations computed from survey data. Partial correlations estimated via precision matrix inversion. '
    f'Edges retained with |r| > {threshold}.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>4. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/03_psychological_network.png'):
        story.append(Image('figures/03_psychological_network.png', width=500, height=400))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Psychological network from partial correlations.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>5. Interpretation</b>', styles['Heading2']))
interp = (
    'Green edges represent positive associations; red dashed edges represent negative associations. '
    'Directly connected nodes have conditional dependence after accounting for all other variables.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')